# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [ ]:
# TODO : Déclarer vos tables Bronze à croiser (table principale + dataset(s) / table(s) secondaire(s))
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/orders",                      # table de faits : 1 ligne par commande
    "order_items": "/lakehouse/bronze/order_items",
    "order_payments": "/lakehouse/bronze/order_payments",
    "order_reviews": "/lakehouse/bronze/order_reviews",
    "customers": "/lakehouse/bronze/customers",
    "products": "/lakehouse/bronze/products",
    "category_translation": "/lakehouse/bronze/category_translation",
    "geolocation": "/lakehouse/bronze/geolocation",
}

assert len(BRONZE_TABLES) > 0, "TODO : Renseignez vos chemins Delta Bronze dans `BRONZE_TABLES`."
bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
for alias, df in bronze_dfs.items():
    print(f"[{alias}] {BRONZE_TABLES[alias]} -> {df.count():,} lignes | colonnes : {df.columns[:8]}")

# TODO : Pointer `df_bronze` sur votre table principale et calculer le diagnostic de valeurs nulles `null_report`
df_bronze = bronze_dfs.get("main")
META_COLS = {"_ingested_at", "_source_dataset", "_source_file"}
profiled_cols = [c for c in df_bronze.columns if c not in META_COLS]
null_report = df_bronze.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in profiled_cols]
)
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)
# Doublons : lignes en trop par rapport à la clé attendue de chaque table
KEYS = {
    "main": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
    "geolocation": ["geolocation_zip_code_prefix"],
}
for alias, keys in KEYS.items():
    df = bronze_dfs[alias]
    extra = df.count() - df.dropDuplicates(keys).count()
    print(f"[{alias:<20}] clé {keys} -> {extra:,} ligne(s) en double")

# Cardinalité par commande : combien de lignes par order_id dans chaque table secondaire ?
for alias in ["order_items", "order_payments", "order_reviews"]:
    df = bronze_dfs[alias]
    print(f"[{alias:<20}] {df.count():,} lignes pour {df.select('order_id').distinct().count():,} commandes distinctes")


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [ ]:
from pyspark.sql import Window

SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"

spark.conf.set("spark.sql.shuffle.partitions", "16")  # jeux de données modestes : 200 partitions seraient inutiles


def normalize_zip(col_name: str):
    """Aligne les codes postaux sur 5 caractères (clé de jointure customers <-> geolocation)."""
    return F.lpad(F.col(col_name).cast("string"), 5, "0")


# 1) Table de faits nettoyée : 1 ligne par commande livrée, avec des dates cohérentes
orders = (
    bronze_dfs["main"]
    .dropDuplicates(["order_id"])
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .filter(F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))
    .select(
        "order_id",
        "customer_id",
        "order_status",
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_ts"),
    )
)
count_before_join = orders.count()

# 2) Sources secondaires ramenées à UNE ligne par clé de jointure (anti-explosion N-à-M)
customers = (
    bronze_dfs["customers"]
    .dropDuplicates(["customer_id"])
    .select(
        "customer_id",
        "customer_unique_id",
        F.initcap(F.trim("customer_city")).alias("customer_city"),
        F.upper(F.trim("customer_state")).alias("customer_state"),
        normalize_zip("customer_zip_code_prefix").alias("customer_zip"),
    )
)

categories = (
    bronze_dfs["category_translation"]
    .select("product_category_name", "product_category_name_english")
    .dropDuplicates(["product_category_name"])
)
products = (
    bronze_dfs["products"]
    .dropDuplicates(["product_id"])
    .select("product_id", "product_category_name")
    .join(F.broadcast(categories), "product_category_name", "left")
    .withColumn(
        "category",
        F.coalesce("product_category_name_english", "product_category_name", F.lit("unknown")),
    )
    .select("product_id", "category")
)

items = (
    bronze_dfs["order_items"]
    .filter(F.col("price") > 0)
    .join(F.broadcast(products), "product_id", "left")
    .groupBy("order_id")
    .agg(
        F.count("*").alias("items_count"),
        F.countDistinct("seller_id").alias("sellers_count"),
        F.round(F.sum("price"), 2).alias("items_total_price"),
        F.round(F.sum("freight_value"), 2).alias("freight_total"),
        F.max_by("category", "price").alias("main_category"),
    )
    .withColumn("main_category", F.coalesce("main_category", F.lit("unknown")))
)

payments = (
    bronze_dfs["order_payments"]
    .filter(F.col("payment_value") > 0)
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("payment_total"),
        F.max("payment_installments").alias("max_installments"),
        F.max_by("payment_type", "payment_value").alias("main_payment_type"),
    )
)

# Plusieurs avis possibles par commande : on garde le plus récent
latest_review = Window.partitionBy("order_id").orderBy(
    F.col("review_answer_timestamp").desc(), F.col("review_id")
)
reviews = (
    bronze_dfs["order_reviews"]
    .filter(F.col("review_score").between(1, 5))
    .withColumn("rn", F.row_number().over(latest_review))
    .filter(F.col("rn") == 1)
    .select("order_id", "review_score")
)

# Référentiel géographique : plusieurs points GPS par code postal -> 1 point moyen (bornes du Brésil)
geo = (
    bronze_dfs["geolocation"]
    .filter(F.col("geolocation_lat").between(-34, 6) & F.col("geolocation_lng").between(-74, -34))
    .withColumn("geo_zip", normalize_zip("geolocation_zip_code_prefix"))
    .groupBy("geo_zip")
    .agg(
        F.round(F.avg("geolocation_lat"), 4).alias("customer_lat"),
        F.round(F.avg("geolocation_lng"), 4).alias("customer_lng"),
    )
)

# 3) Jointures (LEFT JOIN depuis la table de faits) + indicateurs métier
df_silver = (
    orders
    .join(customers, "customer_id", "left")
    .join(F.broadcast(geo), F.col("customer_zip") == F.col("geo_zip"), "left")
    .drop("geo_zip")
    .join(items, "order_id", "left")
    .join(payments, "order_id", "left")
    .join(reviews, "order_id", "left")
    .withColumn("delivery_time_days", F.datediff("order_delivered_ts", "order_purchase_ts"))
    .withColumn("delivery_delay_days", F.datediff("order_delivered_ts", "order_estimated_ts"))
    .withColumn("is_late", F.col("delivery_delay_days") > 0)
    .withColumn("order_total_value", F.round(F.col("items_total_price") + F.col("freight_total"), 2))
    .withColumn("freight_ratio", F.round(F.col("freight_total") / F.col("items_total_price"), 3))
    .withColumn("purchase_month", F.date_format("order_purchase_ts", "yyyy-MM"))
    .withColumn("_silver_updated_at", F.current_timestamp())
)

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé et joint `df_silver`."
df_silver.explain("simple")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
if count_before_join is not None:
    assert silver_count <= count_before_join, (
        f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
    )
print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes (avant jointure : {count_before_join:,})")

# Taux de rapprochement (match rate) : part des commandes retrouvées dans chaque source jointe
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
joined_cols = ["customer_unique_id", "customer_lat", "items_total_price", "payment_total", "review_score"]
silver.agg(
    *[F.round(100 * F.avg(F.col(c).isNotNull().cast("int")), 2).alias(f"match_{c}_%") for c in joined_cols]
).show(truncate=False)


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# Version de référence AVANT nos modifications (créée lors de ce run -> jamais au-delà de la rétention Delta 168h)
base_version = target_delta.history(1).select("version").first()["version"]
base_count = spark.read.format("delta").option("versionAsOf", base_version).load(SILVER_MAIN_PATH).count()

# Lot de 4 commandes réelles (matérialisé pour rester stable après la suppression)
sample = spark.read.format("delta").load(SILVER_MAIN_PATH).orderBy("order_id").limit(4)
batch = spark.createDataFrame(sample.collect(), sample.schema)

# On retire une commande pour simuler une correction a posteriori
late_order_id = batch.first()["order_id"]
target_delta.delete(F.col("order_id") == late_order_id)

# MERGE INTO : met à jour les 3 commandes présentes, réinsère celle qui manque
updates = batch.withColumn("_silver_updated_at", F.current_timestamp())
(
    target_delta.alias("t")
    .merge(updates.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdate(set={"_silver_updated_at": "s._silver_updated_at"})
    .whenNotMatchedInsertAll()
    .execute()
)

count_now = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
print(f"Lignes version {base_version} (avant) : {base_count:,} | version courante : {count_now:,} | commande réinsérée : {late_order_id}")

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"
